# 02 — Golden Dataset QA Validation
Validates every expected value in `evals/golden_dataset.json` directly against DuckDB.
Run this before updating the golden dataset to catch wrong expected values.

In [ ]:
import duckdb
import pandas as pd
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

db = duckdb.connect('../data/spend_intelligence.duckdb', read_only=True)
policy_text = open('../data/procurement_policy.md').read()
print('Connected.')

## q01 — Invoices over EUR 1,000 missing a PO (`NO_PO`)
Original expected: `["14", "735"]` — **both wrong**. Actual count is 21.

In [ ]:
df_q01 = db.execute("""
    SELECT
        ie.invoice_number,
        ie.vendor_name_raw,
        ie.amount,
        ie.currency,
        ie.po_number
    FROM invoices_enriched ie
    JOIN compliance_flags cf ON ie.row_id = cf.row_id
    WHERE cf.flag_type = 'NO_PO'
    ORDER BY ie.amount DESC
""").fetchdf()

print(f"NO_PO count: {len(df_q01)}")
print()
print("Breakdown by currency:")
print(df_q01.groupby('currency')['amount'].agg(['count', 'sum']).round(0))
df_q01.head(10)

## q02 — Total spend with non-ACTIVE vendors
Original expected: `["961", "311", "195", "29"]` — **all wrong**.

Note: no `amount_eur` column in the DB — amounts are in original currency.
A good answer from the system should break down by `vendor_status` and `currency`.

In [ ]:
df_q02 = db.execute("""
    SELECT
        vendor_status,
        currency,
        COUNT(*) AS invoice_count,
        ROUND(SUM(amount), 0) AS total_amount
    FROM invoices_enriched
    WHERE vendor_status NOT IN ('ACTIVE')
      AND vendor_status IS NOT NULL
      AND NOT quarantined
    GROUP BY vendor_status, currency
    ORDER BY vendor_status, currency
""").fetchdf()

print("Non-ACTIVE vendor spend by status + currency:")
print(df_q02.to_string(index=False))
print()

summary = db.execute("""
    SELECT vendor_status, COUNT(*) AS invoice_count, ROUND(SUM(amount), 0) AS total_raw
    FROM invoices_enriched
    WHERE vendor_status NOT IN ('ACTIVE') AND vendor_status IS NOT NULL AND NOT quarantined
    GROUP BY vendor_status
""").fetchdf()
print("Correct expected values (per status, all currencies combined):")
print(summary.to_string(index=False))

## q03 — Who can approve EUR 60,000? (policy-only §3)
Expected: `["50,000", "Finance Director", "250,000"]`

In [ ]:
section3 = re.search(r'(## 3[^\n].*?)(?=## [0-9]|\Z)', policy_text, re.DOTALL)
if section3:
    print(section3.group(0)[:2000])
else:
    print('Section 3 not found — check header format in procurement_policy.md')

## q04 — Duplicate invoice count
Expected: `["10", "duplicate"]` — **correct**.

In [ ]:
df_q04 = db.execute("""
    SELECT flag_type, COUNT(*) AS cnt
    FROM compliance_flags
    WHERE flag_type LIKE '%DUPLICATE%'
    GROUP BY flag_type
""").fetchdf()
print(df_q04)

df_q04_detail = db.execute("""
    SELECT ie.invoice_number, ie.vendor_name_raw, ie.amount, ie.currency, cf.detail
    FROM invoices_enriched ie
    JOIN compliance_flags cf ON ie.row_id = cf.row_id
    WHERE cf.flag_type = 'POTENTIAL_DUPLICATE'
    ORDER BY ie.invoice_number
""").fetchdf()
print(df_q04_detail)

## q05 — Standard payment terms (policy-only §5.2)
Expected: `["60", "days", "net"]`

In [ ]:
section5 = re.search(r'(## 5[^\n].*?)(?=## [0-9]|\Z)', policy_text, re.DOTALL)
if section5:
    print(section5.group(0)[:1500])
else:
    print('Section 5 not found')

## q06 — Invoices with no approver (PENDING_APPROVAL)
Expected: `["26", "pending"]` — **correct**.

In [ ]:
df_q06 = db.execute("""
    SELECT COUNT(*) AS cnt FROM compliance_flags WHERE flag_type = 'PENDING_APPROVAL'
""").fetchdf()
print('PENDING_APPROVAL count:', df_q06['cnt'][0])

# Sanity check: approved_by must be null/empty for all of these
df_q06_check = db.execute("""
    SELECT COALESCE(ie.approved_by, '(null)') AS approved_by, COUNT(*) AS cnt
    FROM invoices_enriched ie
    JOIN compliance_flags cf ON ie.row_id = cf.row_id
    WHERE cf.flag_type = 'PENDING_APPROVAL'
    GROUP BY ie.approved_by
""").fetchdf()
print(df_q06_check)

## q07 — ON_HOLD vendor handling (policy-only §4.1)
Expected: `["escalat", "Procurement Excellence", "4.1"]`

In [ ]:
section4 = re.search(r'(## 4[^\n].*?)(?=## [0-9]|\Z)', policy_text, re.DOTALL)
if section4:
    print(section4.group(0)[:2000])
else:
    print('Section 4 not found')

## q08 — Overdue for approval (OVERDUE_APPROVAL)
Expected: `["24", "overdue"]` — **correct**.

In [ ]:
df_q08 = db.execute("""
    SELECT COUNT(*) AS cnt FROM compliance_flags WHERE flag_type = 'OVERDUE_APPROVAL'
""").fetchdf()
print('OVERDUE_APPROVAL count:', df_q08['cnt'][0])

db.execute("""
    SELECT flag_type, COUNT(*) AS cnt
    FROM compliance_flags
    WHERE flag_type IN ('PENDING_APPROVAL', 'OVERDUE_APPROVAL')
    GROUP BY flag_type
""").fetchdf()

## q09 — Negative amounts / CREDIT_NOTE (hybrid §6.1)
Expected: `["credit", "note", "match", "original"]`

In [ ]:
df_q09 = db.execute("""
    SELECT COUNT(*) AS credit_note_count, ROUND(SUM(amount), 0) AS total
    FROM invoices_enriched
    WHERE is_credit_note = true
""").fetchdf()
print('Credit notes (data):', df_q09)

df_q09_flags = db.execute("""
    SELECT COUNT(*) AS cnt FROM compliance_flags WHERE flag_type = 'CREDIT_NOTE'
""").fetchdf()
print('CREDIT_NOTE flags:', df_q09_flags)

section6 = re.search(r'(## 6[^\n].*?)(?=## [0-9]|\Z)', policy_text, re.DOTALL)
if section6:
    print()
    print(section6.group(0)[:1500])

## q10 — Record retention period (policy-only §8.1)
Expected: `["10", "years"]`

In [ ]:
section8 = re.search(r'(## 8[^\n].*?)(?=## [0-9]|\Z)', policy_text, re.DOTALL)
if section8:
    print(section8.group(0)[:1500])
else:
    print('Section 8 not found')

## Policy-only keyword check
Verifies all expected keywords for q03, q05, q07, q10 exist verbatim in the policy.

In [ ]:
checks = [
    ('q03', ['50,000', 'Finance Director', '250,000'], '§3'),
    ('q05', ['60', 'days', 'net'],                     '§5.2'),
    ('q07', ['escalat', 'Procurement Excellence', '4.1'], '§4.1'),
    ('q10', ['10', 'years'],                            '§8.1'),
]

for qid, keywords, section in checks:
    hits   = [kw for kw in keywords if kw.lower() in policy_text.lower()]
    misses = [kw for kw in keywords if kw.lower() not in policy_text.lower()]
    status = '✅' if not misses else f'⚠️  missing from policy: {misses}'
    print(f"{qid} {section}: [{len(hits)}/{len(keywords)}]  {status}")

## Corrected golden dataset summary
Prints the correct values to paste into `evals/golden_dataset.json`.

In [ ]:
no_po          = db.execute("SELECT COUNT(*) FROM compliance_flags WHERE flag_type='NO_PO'").fetchone()[0]
inactive_count = db.execute("SELECT COUNT(*) FROM invoices_enriched WHERE vendor_status='INACTIVE' AND NOT quarantined").fetchone()[0]
on_hold_count  = db.execute("SELECT COUNT(*) FROM invoices_enriched WHERE vendor_status='ON_HOLD'  AND NOT quarantined").fetchone()[0]
dupes          = db.execute("SELECT COUNT(*) FROM compliance_flags WHERE flag_type='POTENTIAL_DUPLICATE'").fetchone()[0]
pending        = db.execute("SELECT COUNT(*) FROM compliance_flags WHERE flag_type='PENDING_APPROVAL'").fetchone()[0]
overdue        = db.execute("SELECT COUNT(*) FROM compliance_flags WHERE flag_type='OVERDUE_APPROVAL'").fetchone()[0]
credit_notes   = db.execute("SELECT COUNT(*) FROM compliance_flags WHERE flag_type='CREDIT_NOTE'").fetchone()[0]

print("=== Corrected expected_answer_contains for golden_dataset.json ===")
print(f"q01 NO_PO:            count={no_po}   → ['{no_po}']")
print(f"q02 non-ACTIVE:       INACTIVE={inactive_count}, ON_HOLD={on_hold_count}")
print(f"                      → ['INACTIVE', 'ON_HOLD', '{inactive_count}', '{on_hold_count}']")
print(f"q04 duplicates:       count={dupes}   → ['{dupes}', 'duplicate']")
print(f"q06 PENDING_APPROVAL: count={pending}  → ['{pending}', 'pending']")
print(f"q08 OVERDUE_APPROVAL: count={overdue}  → ['{overdue}', 'overdue']")
print(f"q09 CREDIT_NOTE:      count={credit_notes}   (check §6.1 keywords separately)")